# Run the trained 32K SFT model on CPU

Choose **CPU**, enable **Internet**, and attach the safetensors dataset
produced by `nano_dsv41f_export_sft_safetensors_cpu.ipynb`. No optimizer,
training corpus, or separate tokenizer input is needed.

The runtime comes from GitHub and uses the project's Torch CPU implementation
with incremental decoding and DeepSeek V4.1 chat encoding. CPU execution uses
FP32; the saved weights keep their original precision. Start with short prompts
and outputs: the stored 32K context limit does not promise fast CPU inference.
This notebook loads your trained weights; it never initializes a replacement model.


In [ ]:
SOURCE_REF = 'codex/midtrain8k-sft16k'
MODEL_DIR = ''  # blank: locate exactly one exported SFT bundle under /kaggle/input


In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

if any(name in sys.modules for name in ('jax', 'nano_dsv41f')):
    raise RuntimeError('Restart the session before updating the source and dependencies')
os.environ['JAX_PLATFORMS'] = 'cpu'
REPO = Path('/kaggle/working/nano-dsv4.1f')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/xiayicheng3-code/nano-dsv4.1f.git', str(REPO)], check=True)
if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain',
                            '--untracked-files=no'], text=True).strip():
    raise RuntimeError('Preserve tracked edits before updating this checkout')
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', SOURCE_REF], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
if importlib.util.find_spec('torch') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                    '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO / 'requirements-posttrain-inference.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), '--no-deps'], check=True)
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(REPO / 'scripts'))
print('Source commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import json
import torch
from nano_dsv41f.portable_bundle import verify_portable_bundle
from nano_dsv41f.vllm_v41_cpu.api import NanoDeepSeekProtocolBackend

if MODEL_DIR:
    model_root = Path(MODEL_DIR)
else:
    bundles = list(Path('/kaggle/input').rglob('export_manifest.json'))
    if len(bundles) != 1:
        raise ValueError('Attach one exported bundle or set MODEL_DIR to its directory')
    model_root = bundles[0].parent
report = verify_portable_bundle(model_root)
config = json.loads((model_root / 'config.json').read_text())
MAX_CONTEXT = int(config['max_position_embeddings'])
torch.set_num_threads(max(1, (os.cpu_count() or 2) - 1))
backend = NanoDeepSeekProtocolBackend.from_pretrained(model_root, dtype=torch.float32)
print('Loaded SFT checkpoint:', report['checkpoint'])
print('Context:', MAX_CONTEXT, 'RoPE:', config['nano_config']['attention']['rope'])

def check_context(prompt_ids, max_tokens):
    if max_tokens <= 0 or len(prompt_ids) + max_tokens > MAX_CONTEXT:
        raise ValueError(f'Prompt + output must fit {MAX_CONTEXT} tokens; shorten the request or reset_chat()')

def complete(prompt, *, max_tokens=64, temperature=0.0):
    check_context(backend.tokenizer.encode(prompt), max_tokens)
    return backend.complete_raw(prompt, max_tokens=max_tokens, temperature=temperature)


In [ ]:
# Multi-turn DeepSeek V4.1 Chat Completions helper.
import json

history = []
VALID_REASONING_EFFORTS = {"minimal", "low", "medium", "high", "xhigh", "max"}

def reset_chat():
    history.clear()

def chat(message, *, thinking=False, reasoning_effort="high", max_tokens=128):
    if reasoning_effort not in VALID_REASONING_EFFORTS:
        raise ValueError(
            f"reasoning_effort must be one of {sorted(VALID_REASONING_EFFORTS)}"
        )
    request_messages = history + [{"role": "user", "content": message}]
    payload = {
        "model": backend.model_name,
        "messages": request_messages,
        "thinking": {"type": "enabled" if thinking else "disabled"},
        "reasoning_effort": reasoning_effort,
        "max_tokens": max_tokens,
        "temperature": 1.0,
        "top_p": 0.95,
        "stream": False,
    }
    from nano_dsv41f.vllm_v41_cpu.api import prepare_protocol_request
    prepared = prepare_protocol_request("chat_completions", payload)
    prompt_ids = backend.tokenizer.encode_conversation(prepared.conversation_request.conversation)
    check_context(prompt_ids, max_tokens)
    _, raw = backend.complete_protocol("chat_completions", payload)
    response = json.loads(raw)
    assistant = response["choices"][0]["message"]
    history.append({"role": "user", "content": message})
    saved = {"role": "assistant", "content": assistant.get("content") or ""}
    if assistant.get("reasoning_content"):
        saved["reasoning_content"] = assistant["reasoning_content"]
    if assistant.get("tool_calls"):
        saved["tool_calls"] = assistant["tool_calls"]
    history.append(saved)
    return assistant

def print_reply(reply):
    reasoning = reply.get("reasoning_content")
    if reasoning:
        print("Reasoning:\n" + reasoning + "\n")
    print("Assistant:\n" + (reply.get("content") or ""))


In [ ]:
# A short first response checks model loading, chat encoding, and cached generation.
print_reply(chat('Hello! Introduce yourself in one sentence.', max_tokens=32))
reset_chat()


## Chat here

The next cell creates a small in-notebook chat control. Type a message and press
**Send**; the conversation history is preserved until you press **Reset**.
Thinking mode and DeepSeek reasoning effort can be changed from the controls.


In [ ]:
# Kaggle/Jupyter-native chat controls: no public tunnel or separate web app needed.
import ipywidgets as widgets
from IPython.display import display

message_box = widgets.Textarea(
    placeholder="Type a message…",
    layout=widgets.Layout(width="100%", height="90px"),
)
thinking_box = widgets.Checkbox(value=False, description="Thinking")
effort_box = widgets.Dropdown(
    options=["minimal", "low", "medium", "high", "xhigh", "max"],
    value="high",
    description="Effort:",
)
max_tokens_box = widgets.BoundedIntText(
    value=128,
    min=1,
    max=1024,
    step=1,
    description="Max tokens:",
)
send_button = widgets.Button(description="Send", button_style="primary")
reset_button = widgets.Button(description="Reset")
chat_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd"))

def _send(_):
    message = message_box.value.strip()
    if not message:
        return
    message_box.value = ""
    with chat_output:
        print(f"You: {message}")
        try:
            reply = chat(
                message,
                thinking=thinking_box.value,
                reasoning_effort=effort_box.value,
                max_tokens=max_tokens_box.value,
            )
            reasoning = reply.get("reasoning_content")
            if reasoning:
                print(f"Reasoning: {reasoning}")
            print(f"Assistant: {reply.get('content') or ''}\n")
        except Exception as exc:
            print(f"Error: {exc}\n")

def _reset(_):
    reset_chat()
    chat_output.clear_output()
    with chat_output:
        print("Conversation reset.\n")

send_button.on_click(_send)
reset_button.on_click(_reset)
display(
    widgets.VBox(
        [
            message_box,
            widgets.HBox([thinking_box, effort_box, max_tokens_box]),
            widgets.HBox([send_button, reset_button]),
            chat_output,
        ]
    )
)


## Or call the helper directly

```python
print_reply(chat("Why is sparse attention useful?"))
print_reply(chat("Now explain that more simply."))  # keeps the same history
```

For DeepSeek-style reasoning mode, use the API's named reasoning levels
(`minimal`, `low`, `medium`, `high`, `xhigh`, or `max`):

```python
print_reply(
    chat(
        "Solve 37*43 carefully.",
        thinking=True,
        reasoning_effort="high",
        max_tokens=256,
    )
)
```

Use `reset_chat()` to start a fresh conversation.


## Raw completion and tool output

`complete("The capital of France is", max_tokens=32)` uses the prompt directly.
`chat(...)` applies the V4.1 conversation protocol and maintains history.
Both functions check prompt plus requested output against the saved context limit.
For tool-call inspection, examine the returned reply dictionary; this notebook
does not execute generated tool calls. Use `reset_chat()` for a new conversation.
The optional local API cell below serves the same loaded model.


In [ ]:
# Optional: expose the same model through local OpenAI/Anthropic-compatible HTTP routes.
# Run this cell only when you want an API server inside the Kaggle session.
# from nano_dsv41f.vllm_v41_cpu.api import create_app
# import uvicorn
# app = create_app(backend)
# uvicorn.run(app, host="127.0.0.1", port=8000)
